# Silver — espelho governado do bronze

A regra da casa, e ela não é negociável:

A silver é o espelho do bronze com governança aplicada. **Mesmo nome de tabela, mesmo grão, mesma contagem de linhas.**

## Permitido na silver

* **tipagem**: string vira `TIMESTAMP`, `INT`, `DATE`
* **legibilidade**: quebrar timestamp em data e hora
* **metadados**: `CURRENT` em toda coluna, tags na tabela
* **unificação**: dois cadastros do mesmo assunto, com a origem por registro
* **aritmética pura**: `atraso = real - previsto`

## Proibido na silver

* **filtro / `WHERE` de negócio** ❌
* **`GROUP BY` / agregação** ❌
* **limpar flag, classificação** ❌


**Por quê?** Porque a silver precisa servir várias análises, e toda linha que ela descarta é uma pergunta que ninguém mais vai conseguir fazer. Filtro fecha porta.

O teste para qualquer coluna nova: *isso embute uma decisão de negócio?* `atraso_partida_min = partida_real - partida_prevista` é subtração - silver. `partida_pontual = atraso <= 15>` embute o número **15**, que é decisão de negócio e  muda por cliente - gold.


# 1. O que precisa ser consertado na tipagem

Antes de escrever o `CAST`, medir. Duas armadilhas escondidas no bronze:

In [0]:
SELECT
    COUNT(*)                                                           AS linhas,
    SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END)             AS partida_real_null_de_verdade,
    SUM(CASE WHEN partida_real = 'null' THEN 1 ELSE 0 END)            AS partida_real_string_null,
    SUM(CASE WHEN partida_prevista = 'null' THEN 1 ELSE 0 END)        AS partida_prevista_string_null,
    SUM(CASE WHEN partida_prevista LIKE '%:%' THEN 1 ELSE 0 END)      AS com_fracao_de_segundo,
    SUM(CASE WHEN partida_prevista LIKE '"%' THEN 1 ELSE 0 END)       AS com_aspas_duplas
FROM voebem.bronze.vra

**Armadilha 1 - a ausência veio como a string `null`.** Quatro caracteres de texto. `WHERE partida_real IS NULL` devolve **zero** numa tabela onde 29 mil voos não têm horário real. Correção: `nullif(coluna, 'null')` **antes** do cast.

**Armadilha 2 - dois formatos de timestamp no mesmo arquivo.** A maioria vem `2026-01-27 19:45:00`, mas ~80 mil linhas vêm com fração de segundo de 9 casas. Um `to_timestamp(col, 'yyyy-MM-dd HH:mm:ss')` fixo devolveria NULL para 8% da base, em silêncio. O `try_cast(... AS TIMESTAMP)` aceita os dois formatos, e o `try_` garante que um formato inválido retorna NULL em vez de explodir a query inteira.

**Armadilha 3 - aspas duplas literais dentro dos timestamps.** Os valores vêm como `"2026-01-27 19:45:00"` (com aspas dentro da string). O `try_cast` direto falharia em 100% dos casos porque `"2026-01-27..."` não é formato válido de timestamp. Correção: `regexp_replace(coluna, '"', '')` **antes** do `nullif` e `try_cast`, para remover as aspas.

# 2. silver.vra - o espelho

Repare no que **não** existe nesta query: nenhum `WHERE`, nenhum `GROUP BY`, nenhum `DISTINCT`, nenhum `JOIN`. É um `SELECT` de projeção sobre o bronze inteiro.

E repare nas três colunas do fim: `atraso_partida_min`, `atraso_chegada_min`, `minutos_recuperados`. São subtrações entre colunas da própria linha. Não têm limiar, não classificam nada, não escondem número mágico - e, principalmente, não impedem análise nenhuma. Por isso podem morar aqui.


In [0]:
CREATE SCHEMA IF NOT EXISTS voebem.silver;

CREATE OR REPLACE TABLE voebem.silver.vra AS
WITH tipado AS (
    SELECT
        icao_empresa_aerea,
        numero_voo,
        codigo_autorizacao_di,
        codigo_tipo_linha,
        icao_aerodromo_origem,
        icao_aerodromo_destino,
        try_cast(regexp_replace(nullif(partida_prevista, 'null'), '"', '') AS TIMESTAMP) AS partida_prevista,
        try_cast(regexp_replace(nullif(partida_real, 'null'), '"', '') AS TIMESTAMP) AS partida_real,
        try_cast(regexp_replace(nullif(chegada_prevista, 'null'), '"', '') AS TIMESTAMP) AS chegada_prevista,
        try_cast(regexp_replace(nullif(chegada_real, 'null'), '"', '') AS TIMESTAMP) AS chegada_real,
        situacao_voo,
        nullif(codigo_justificativa, 'N/A') AS codigo_justificativa
    FROM voebem.bronze.vra
)
SELECT
    icao_empresa_aerea,
    numero_voo,
    codigo_autorizacao_di,
    codigo_tipo_linha,
    icao_aerodromo_origem,
    icao_aerodromo_destino,
    partida_prevista,
    partida_real,
    chegada_prevista,
    chegada_real,
    situacao_voo,
    codigo_justificativa,
    TIMESTAMPDIFF(MINUTE, partida_prevista, partida_real) AS atraso_partida_min,
    TIMESTAMPDIFF(MINUTE, chegada_prevista, chegada_real) AS atraso_chegada_min,
    TIMESTAMPDIFF(MINUTE, partida_real, chegada_real) - TIMESTAMPDIFF(MINUTE, partida_prevista, chegada_prevista) AS minutos_recuperados
FROM tipado

In [0]:
-- Validar a tabela silver: contagem, conversões e cálculos
SELECT 
    icao_empresa_aerea,
    numero_voo,
    icao_aerodromo_origem,
    icao_aerodromo_destino,
    partida_prevista,
    partida_real,
    atraso_partida_min,
    chegada_prevista,
    chegada_real,
    atraso_chegada_min,
    minutos_recuperados
FROM voebem.silver.vra
WHERE atraso_partida_min IS NOT NULL
ORDER BY ABS(atraso_partida_min) DESC
LIMIT 10

In [0]:
-- Estatísticas de qualidade: contagem, conversões bem-sucedidas e cálculos
SELECT 
    'Total de linhas' AS metrica,
    COUNT(*) AS valor
FROM voebem.silver.vra

UNION ALL

SELECT 
    'Bronze → Silver (espelho 1:1)' AS metrica,
    COUNT(*) AS valor
FROM voebem.bronze.vra

UNION ALL

SELECT 
    'Timestamps convertidos (partida_prevista)' AS metrica,
    SUM(CASE WHEN partida_prevista IS NOT NULL THEN 1 ELSE 0 END) AS valor
FROM voebem.silver.vra

UNION ALL

SELECT 
    'Timestamps convertidos (partida_real)' AS metrica,
    SUM(CASE WHEN partida_real IS NOT NULL THEN 1 ELSE 0 END) AS valor
FROM voebem.silver.vra

UNION ALL

SELECT 
    'Cálculos de atraso disponíveis' AS metrica,
    SUM(CASE WHEN atraso_partida_min IS NOT NULL THEN 1 ELSE 0 END) AS valor
FROM voebem.silver.vra

UNION ALL

SELECT 
    'Taxa de conversão (%)' AS metrica,
    ROUND(SUM(CASE WHEN partida_prevista IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS valor
FROM voebem.silver.vra

In [0]:
-- Verificar a distribuição temporal dos dados (range de datas)
SELECT 
    MIN(partida_prevista) AS primeira_partida,
    MAX(partida_prevista) AS ultima_partida,
    DATEDIFF(MAX(partida_prevista), MIN(partida_prevista)) AS dias_de_cobertura,
    COUNT(DISTINCT DATE(partida_prevista)) AS dias_unicos_com_dados
FROM voebem.silver.vra
WHERE partida_prevista IS NOT NULL

In [0]:
-- Análise da distribuição de atrasos: adiantados, pontuais e atrasados
SELECT 
    CASE 
        WHEN atraso_partida_min IS NULL THEN 'Sem dados'
        WHEN atraso_partida_min < -15 THEN 'Muito adiantado (>15min)'
        WHEN atraso_partida_min BETWEEN -15 AND 0 THEN 'Adiantado (0-15min)'
        WHEN atraso_partida_min BETWEEN 1 AND 15 THEN 'Levemente atrasado (1-15min)'
        WHEN atraso_partida_min BETWEEN 16 AND 60 THEN 'Atrasado (16-60min)'
        WHEN atraso_partida_min > 60 THEN 'Muito atrasado (>60min)'
    END AS categoria_atraso,
    COUNT(*) AS quantidade,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentual
FROM voebem.silver.vra
GROUP BY 1
ORDER BY 2 DESC

In [0]:
-- Detectar outliers e dados suspeitos que precisam de investigação
SELECT 
    'Atrasos extremos (> 24h)' AS tipo_outlier,
    COUNT(*) AS quantidade
FROM voebem.silver.vra
WHERE ABS(atraso_partida_min) > 1440

UNION ALL

SELECT 
    'Voos com chegada antes da partida' AS tipo_outlier,
    COUNT(*) AS quantidade
FROM voebem.silver.vra
WHERE partida_real IS NOT NULL 
  AND chegada_real IS NOT NULL
  AND chegada_real < partida_real

UNION ALL

SELECT 
    'Voos com duração > 24h' AS tipo_outlier,
    COUNT(*) AS quantidade
FROM voebem.silver.vra
WHERE partida_real IS NOT NULL 
  AND chegada_real IS NOT NULL
  AND TIMESTAMPDIFF(HOUR, partida_real, chegada_real) > 24

UNION ALL

SELECT 
    'Recuperação de tempo suspeita (> 2h)' AS tipo_outlier,
    COUNT(*) AS quantidade
FROM voebem.silver.vra
WHERE ABS(minutos_recuperados) > 120

In [0]:
-- Top 10 empresas por volume de voos
SELECT 
    icao_empresa_aerea,
    COUNT(*) AS total_voos,
    COUNT(CASE WHEN atraso_partida_min IS NOT NULL THEN 1 END) AS voos_com_horario,
    ROUND(AVG(CASE WHEN atraso_partida_min IS NOT NULL THEN atraso_partida_min END), 1) AS atraso_medio_min,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentual_do_total
FROM voebem.silver.vra
GROUP BY icao_empresa_aerea
ORDER BY total_voos DESC
LIMIT 10

In [0]:
-- Top 15 rotas mais movimentadas
SELECT 
    CONCAT(icao_aerodromo_origem, ' → ', icao_aerodromo_destino) AS rota,
    COUNT(*) AS total_voos,
    COUNT(CASE WHEN atraso_partida_min > 0 THEN 1 END) AS voos_atrasados,
    ROUND(COUNT(CASE WHEN atraso_partida_min > 0 THEN 1 END) * 100.0 / 
          NULLIF(COUNT(CASE WHEN atraso_partida_min IS NOT NULL THEN 1 END), 0), 1) AS perc_atrasados,
    ROUND(AVG(CASE WHEN atraso_partida_min IS NOT NULL THEN atraso_partida_min END), 1) AS atraso_medio_min
FROM voebem.silver.vra
GROUP BY icao_aerodromo_origem, icao_aerodromo_destino
ORDER BY total_voos DESC
LIMIT 15